In [1]:
import os
import importlib
# os.environ["CUDA_VISIBLE_DEVICES"]="2,3"
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM, AutoModel
from datasets import load_dataset
import torch
#from sentence_transformers import SentenceTransformer, InputExample, losses
#from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator, SimilarityFunction
from torch.utils.data import DataLoader
from datasets import Dataset
import pandas as pd
from collections import defaultdict
import re
import numpy as np
from tqdm import tqdm

data_dir = '/raid/deallab/SF_RAG_Data/ASQA'
# data_dir = '../data'

device1 = 'cuda:0'
device2 = 'cuda:1'

gen_model_id = 'meta-llama/Meta-Llama-3.1-8B-Instruct'
# gen_model_id = 'mistralai/Mistral-7B-Instruct-v0.3'

split_token = '<|end_header_id|>'
end_token = '<|eot_id|>'

# split_token = '[/INST]'
# end_token = '</s>'

from evaluation import evaluate
import prompts
importlib.reload(prompts)

/home/dataconv/anaconda3/envs/sf_rag_djk/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<module 'prompts' from '/home/dataconv/deallab/djk/sf_rag/sf_rag/model/prompts.py'>

In [2]:
#load embeddings
embedd_test_path = f'{data_dir}/test/embedd_test.npy'
evidence_embeddings = np.load(embedd_test_path)
print(evidence_embeddings.shape)
evidence_embeddings = torch.from_numpy(evidence_embeddings).to(device1)

#load evidence
evidence_test_path = f'{data_dir}/test/evidence_test.csv'
evidence_df = pd.read_csv(evidence_test_path)

#load qa data
qa_df=pd.read_csv(f'{data_dir}/test/qa_test.csv') #data=df[['question','long_answers']] # questions=data['question'] #references = [row.to_dict() for i, row in df.iterrows() if i < len(questions)]
qa_df.head()

(11657, 4096)


,id,sample_id,question,follow_up_questions,long_answers,short_answers
0,f743f676-48f8-42c6-ab8e-e4cd0a0542ce,-7013890438520559398,Who has the highest goals in world football?,"[""Who has the highest goals in men's world int...","[""Ali Dael has the highest goals in men's worl...","[['Daei', 'Ali Daei'], ['Bican', 'Josef Bican'..."
1,40f108d2-081b-444a-b905-21dc7513628b,7089015503030534342,Who is the original artist of sound of silence?,['Who is the original artist of sound of silen...,[' The original artist of the song sound of si...,"[['Simon & Garfunkel', 'Paul Simon and Art Gar..."
2,71925ca1-fcc5-4856-94cb-da3036123ca0,8793099883447006698,When was the first apple i phone made?,"['When was the first apple i phone released?',...",['The iPhone beta was created in 2004 to test ...,"[['June 29, 2007'], ['2004'], ['June 29, 2007...."
3,f9c40e99-b832-4f71-9d41-f821896e288c,-881464876144297194,Who played the weasley brothers in harry potter?,['Who played Bill weasley in Harry Potter and...,['Rupert Grint played Ron Weasley in all the H...,"[['Richard Fish'], ['Chris Rankin'], ['James P..."
4,2031b0f8-f0a1-4d5d-bed6-f706d1511d42,1650309494326541834,How many state parks are there in virginia?,['How many state parks are there in virginia i...,['When the Virginia state park system was form...,"[['six'], ['38'], ['6'], ['38']]"


In [3]:
#load quantized model
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_storage=torch.bfloat16,
)

# load model with tokenizer
model = AutoModel.from_pretrained(
    'nvidia/NV-Embed-v2', 
    trust_remote_code=True,
    quantization_config = bnb_config,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage =True,
)
model.eval()

Loading checkpoint shards: 100%|██████████| 4/4 [00:11<00:00,  2.88s/it]


NVEmbedModel(
  (latent_attention_model): LatentAttentionModel(
    (cross_attend_blocks): ModuleList(
      (0): PreNorm(
        (fn): Attention(
          (to_q): Linear4bit(in_features=4096, out_features=32768, bias=False)
          (to_kv): Linear4bit(in_features=4096, out_features=65536, bias=False)
          (to_out): Linear4bit(in_features=32768, out_features=4096, bias=False)
        )
        (norm): LayerNorm((4096,), eps=1e-05, elementwise_affine=True)
        (norm_context): LayerNorm((4096,), eps=1e-05, elementwise_affine=True)
      )
      (1): PreNorm(
        (fn): FeedForward(
          (net): Sequential(
            (0): Linear4bit(in_features=4096, out_features=32768, bias=True)
            (1): GEGLU()
            (2): Linear4bit(in_features=16384, out_features=4096, bias=True)
          )
        )
        (norm): LayerNorm((4096,), eps=1e-05, elementwise_affine=True)
      )
    )
  )
  (embedding_model): BidirectionalMistralModel(
    (embed_tokens): Embedding(

In [4]:
#load tokenizer
tokenizer_gen = AutoTokenizer.from_pretrained(gen_model_id)
tokenizer_gen.pad_token = tokenizer_gen.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    # bnb_4bit_quant_type="nf4",
    # bnb_4bit_compute_dtype=torch.bfloat16,
    # bnb_4bit_use_double_quant=True,
    # bnb_4bit_quant_storage=torch.bfloat16,
)

model_gen = AutoModelForCausalLM.from_pretrained(
    gen_model_id,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map= 'auto'
)
model_gen.eval()

Loading checkpoint shards: 100%|██████████| 4/4 [00:16<00:00,  4.13s/it]


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaSdpaAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps

In [5]:
# retrive docs from the document embeddings
def retrieve_documents(query):
    max_length = 1024
    
    #query prefix
    task_name_to_instruct = {"example": "Given a question, retrieve passages that answer the question",}
    query_prefix = "Instruct: "+task_name_to_instruct["example"]+"\nQuery: "
    
    query_embedding = model.encode([query],instruction=query_prefix, max_length=max_length).to(device1)

    # query_embedding = query_embedding.unsqueeze(0)
    #print(query_embedding)
    similarities = torch.nn.functional.cosine_similarity(query_embedding, evidence_embeddings)

    top_results = similarities.argsort(descending=True)[:10].cpu().detach().numpy()
    #print(top_results)
    res=[evidence_df.loc[idx, 'text'] for idx in top_results if idx < len(evidence_df)]
        
    return top_results, res

In [6]:
def evaluate_docs(query, docs):
    # print(f"Query : {query}")
    # print("-"*100)
    outs = []
    for idx, doc in enumerate(docs):
        #print(f"Rank {idx} : {doc}")
        
        input= f'''
        Query: {query}
        Doc:  {doc}
        '''

        messages = [
            {"role":"user", 'content':prompts.PROMPT['eval_doc_instr']},
            {"role":"assistant", 'content':prompts.PROMPT['eval_doc_answ1']},
            {"role":"user", 'content':prompts.PROMPT['eval_doc_ex2']},
            {"role":"assistant", 'content':prompts.PROMPT['eval_doc_answ2']},
            {"role":"user", 'content':prompts.PROMPT['eval_doc_ex3']},
            {"role":"assistant", 'content':prompts.PROMPT['eval_doc_answ3']},
            {"role":"user", 'content':input}, 
        ]
        #apply tokenizter + generate eval
        inputs = tokenizer_gen.apply_chat_template(messages, return_tensors="pt", truncation=True).to(device1)
        
        attention_mask = (inputs != tokenizer_gen.pad_token_id).long().to(device1)
        
        outputs = model_gen.generate(inputs, attention_mask=attention_mask, pad_token_id=tokenizer_gen.pad_token_id, max_new_tokens=128)
        generated_text = tokenizer_gen.decode(outputs[0]).split(split_token)[-1].replace(end_token, '').strip('\n')
        
        if '#relevant' in generated_text:
            outs.append(doc)
    
    return outs

In [7]:
def make_new_query(query,context):
    
    input= f'''
    Original Query: {query}
    Context information: {context}
    '''
    
    messages = [
        {"role":"user", 'content':prompts.PROMPT['refine_query_instr']},
        {"role":"assistant", 'content':prompts.PROMPT['refine_query_answ1']},
        {"role":"user", 'content':prompts.PROMPT['refine_query_ex2']},
        {"role":"assistant", 'content':prompts.PROMPT['refine_query_answ2']},
        {"role":"user", 'content':input},
    ]
    inputs = tokenizer_gen.apply_chat_template(messages, return_tensors="pt", truncation=True).to(device1)
    
    attention_mask = (inputs != tokenizer_gen.pad_token_id).long().to(device1)
    
    outputs = model_gen.generate(inputs, attention_mask=attention_mask, pad_token_id=tokenizer_gen.pad_token_id, max_new_tokens=512)
    generated_text = tokenizer_gen.decode(outputs[0]).split(split_token)[-1].replace(end_token, '').strip('\n')
    generated_text  = generated_text.strip('[]').split('\n')
    
    print(generated_text)
    return generated_text

In [8]:
# def preprocessing(new_questions):
#     return list(((new_questions.split("['")[1]).split("']")[0]).split("',\n    '"))

In [9]:
# preprocessing(make_new_query(query, rel_docs))

In [10]:
def make_new_answer(query,context):
    
    context_str = '\n'.join(context)
    
    input= f'''
    Original Query: {query}
    Context information: {context_str}
    '''
    
    messages = [
        {"role":"user", 'content':prompts.PROMPT['new_answer_instr']},
        {"role":"assistant", 'content':prompts.PROMPT['new_answer_answ1']},
        {"role":"user", 'content':prompts.PROMPT['new_answer_ex2']},
        {"role":"assistant", 'content':prompts.PROMPT['new_answer_answ2']},
        {"role":"user", 'content':input},
    ]
    inputs = tokenizer_gen.apply_chat_template(messages, return_tensors="pt", truncation=True).to(device1)
    
    attention_mask = (inputs != tokenizer_gen.pad_token_id).long().to(device1)
    print(len(inputs[0]))
    outputs = model_gen.generate(inputs, attention_mask=attention_mask, pad_token_id=tokenizer_gen.pad_token_id, max_new_tokens=256)
    generated_text = tokenizer_gen.decode(outputs[0]).split(split_token)[-1].replace(end_token, '').strip('\n')
    
    print(f'New Answer: {generated_text}')
    return generated_text

In [11]:
# import re

# def summarize_answers(question, answers):
#     input= f'''
#     Query: {query}
#     Context information: {answers}
#     '''
    
#     messages = [
#         {"role":"user", 'content':prompts.PROMPT['final_answer_instr']},
#         {"role":"assistant", 'content':prompts.PROMPT['final_answer_answ1']},
#         {"role":"user", 'content':{input}},
#     ]

#     #tokenizer prompt
#     input_ids = tokenizer_gen.apply_chat_template(messages, return_tensors="pt", truncation=True).to(device)

#     attention_mask = (input_ids != tokenizer_gen.pad_token_id).long().to(device)

#     out = model_gen.generate(input_ids, attention_mask=attention_mask, pad_token_id=tokenizer_gen.pad_token_id, max_new_tokens = 512)
#     res = tokenizer_gen.decode(out[0]).split('<|end_header_id|>')[-1] 
#     candidate = [re.sub('\n|<\|eot_id\|>', '', res)]
#     return candidate

In [12]:
# data=qa_df[['question','long_answers']]
# questions=data['question']

In [13]:
# references = [row.to_dict() for i, row in qa_df.iterrows() if i < len(questions)]

In [14]:
# references[0]

In [15]:
def initial_answer(query, docs):
    prompt = """
    Context information is below.
    ---------------------
    {0}
    ---------------------
    Given the context information and not prior knowledge, answer the query.
    Query: {1}
    Answer:
    """.format('\n'.join(docs), query)
    input_ids = tokenizer_gen.apply_chat_template([{"role":'user', "content":prompt}], return_tensors='pt').to(device2)

    attention_mask = (input_ids != tokenizer_gen.pad_token_id).long().to(device2)

    out = model_gen.generate(input_ids, attention_mask=attention_mask, pad_token_id=tokenizer_gen.pad_token_id, max_new_tokens = 512)
    res = tokenizer_gen.decode(out[0]).split('<|end_header_id|>')[-1] 
    text = re.sub('\n|<\|eot_id\|>', '', res)
    return text

In [16]:
def final_ans(query,answer, qa_pairs):
    # prompt = f"""
    # Context information is below.
    # ---------------------
    # {answers}
    # ---------------------
    # Given the context information and not prior knowledge, 
    # Answer questions that have multiple correct answers based on multiple interpretations, including multiple answers.
    # Query: {query}
    # Answer:
    # """
    
    # input_ids = tokenizer_gen.apply_chat_template([{"role":'user', "content":prompt}], return_tensors='pt').to(device1)
    qa_sample = '''Follow-up Query{i}: {q}
    Context: {context}
    '''
    qa_string = '\n'.join([qa_sample.format(i=i, q=q, context=a) for i, (q, a) in enumerate(qa_pairs)])
    
    input= f'''
    Initial Query: {query}
    Context: {answer}
    {qa_string}
    '''
    print(f'Final Answ Input:{input}')
    messages = [
        {"role":"user", 'content':prompts.PROMPT['final_answer_instr']},
        {"role":"assistant", 'content':prompts.PROMPT['final_answer_answ1']},
        {"role":"user", 'content':input},
    ]

    #tokenizer prompt
    input_ids = tokenizer_gen.apply_chat_template(messages, return_tensors="pt", truncation=True).to(device2)

    attention_mask = (input_ids != tokenizer_gen.pad_token_id).long().to(device2)

    out = model_gen.generate(input_ids, attention_mask=attention_mask, pad_token_id=tokenizer_gen.pad_token_id, max_new_tokens = 512)
    res = tokenizer_gen.decode(out[0]).split(split_token)[-1] 
    candidate = re.sub(end_token, '', res)
    #print(candidate)
    return candidate

In [17]:
from evaluation import evaluate
from collections import defaultdict

# sf_rag=dict()
# perplexity_df=pd.DataFrame()
scores_list=[]
stop_iteration = 90
new_answers_dic=defaultdict(list)
stop_iteration = 90
for idx, row in tqdm(qa_df.iterrows(), total=min(len(qa_df), stop_iteration)):
    if idx == stop_iteration: break
    query = row['question']
    
    #retrieve relevant docs
    ids, docs = retrieve_documents(query)
    init_answer=initial_answer(query, docs)
    print(f'Initial Answer: {init_answer}')
    
    rel_docs = evaluate_docs(query, docs)
    rel_answer = make_new_answer(query, rel_docs)
    print(f'Relevant Answer: {rel_answer}')
    
    # generate new queries
    new_queries=make_new_query(query, rel_docs)
    
    #iterate over new docs
    qa_pairs = []
    for i, new_query in tqdm(enumerate(new_queries)):
        if i == 8: break #brak after x follow-up question 
        
        # retrieve relevant docs
        ids, new_docs=retrieve_documents(new_query)
        new_rel_docs=evaluate_docs(new_query, new_docs)
        if not new_rel_docs: continue
        new_answer = make_new_answer(new_query, new_rel_docs)
        qa_pairs.append((new_query, new_answer))

    # generate final answer
    add_answer=final_ans(query, init_answer+rel_answer, qa_pairs)
    print(f'candidate: {init_answer+rel_answer+add_answer}')
    # print(references[i])
    scores=evaluate([init_answer+rel_answer+add_answer],[row.to_dict()])
    print(scores)
    scores_list.append(scores)
    scores_df=pd.DataFrame(scores_list)
    scores_df.to_csv('./results/self-refine_results.csv', index=False)
    
scores_df=pd.DataFrame(scores_list)
print(scores_df.mean())
scores_df.to_csv('./results/self-refine_results.csv', index=False)

  0%|          | 0/90 [00:00<?, ?it/s]/home/dataconv/.cache/huggingface/modules/transformers_modules/nvidia/NV-Embed-v2/7604d305b621f14095a1aa23d351674c2859553a/modeling_nvembed.py:349: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  'input_ids': torch.tensor(batch_dict.get('input_ids').to(batch_dict.get('input_ids')).long()),
/home/dataconv/anaconda3/envs/sf_rag_djk/lib/python3.10/contextlib.py:103: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)
  0%|          | 0/90 [00:03<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 4.69 GiB. GPU 4 has a total capacity of 3.81 GiB of which 812.00 MiB is free. Including non-PyTorch memory, this process has 2.86 GiB memory in use. Of the allocated memory 2.73 GiB is allocated by PyTorch, and 68.55 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
#  20%|██        | 1/5 [01:19<05:16, 79.10s/it]
# ['Based on the provided context information, there are multiple answers to the query "Who has the highest goals in world football?" depending on the interpretation.1. **Cristiano Ronaldo**: With 133 international goals, Cristiano Ronaldo holds the record for the highest number of goals scored in international football.2. **Cristiano Ronaldo (in European football)**: Ronaldo also holds the record for the highest number of goals scored in European football, with 85 international goals.3. **Cristiano Ronaldo (in European Championship)**: He is the first player to score 14 goals at the European Championships.4. **Cristiano Ronaldo (in UEFA Nations League)**: Ronaldo is the top scorer in the inaugural UEFA Nations League, with 5 goals.5. **Pelé**: He was the first player from South America to score at least 50 international goals and went on to score 77 international goals in 92 matches.6. **Mokhtar Dahari**: He broke the record for the highest international goalscorer, scoring 89 goals for Malaysia in 142 international appearances.7. **Imre Schlosser**: He was the first player to score 50 international goals and held the record for 26 years until Ferenc Puskás broke it.8. **Ferenc Puskás**: He broke the record for the highest international goalscorer, scoring 84 goals in his international career.9. **Vivian Woodward**: He was the fastest to achieve the feat of 50 international goals, scoring his 50th goal in his 32nd official international match.10. **Lionel Messi**: He became the third player to reach and pass the milestone of 100 international goals, as well as the first South American to achieve the feat.These are just a few examples of players who have achieved significant milestones in international football.']
# {'rougeLsum': 27.368421052631582, 'length': 264.0, 'str_em': 0.0, 'ovscore': 0.0}
#  40%|████      | 2/5 [02:18<03:22, 67.38s/it]
# ['The original artist of "The Sound of Silence" is Simon & Garfunkel, specifically Paul Simon, who wrote the song, and Art Garfunkel, who sang the melody.']
# {'rougeLsum': 41.463414634146346, 'length': 26.0, 'str_em': 66.66666666666666, 'ovscore': 52.57592264788534}
#  60%|██████    | 3/5 [03:09<02:00, 60.04s/it]
# ['The development of the first Apple iPhone began in 2005, and it was officially announced on January 9, 2007.']
# {'rougeLsum': 28.915662650602407, 'length': 19.0, 'str_em': 0.0, 'ovscore': 0.0}
#  80%|████████  | 4/5 [04:01<00:56, 56.98s/it]
# ['The Weasley brothers were portrayed by James and Oliver Phelps, who played Fred and George Weasley respectively.']
# {'rougeLsum': 19.607843137254903, 'length': 17.0, 'str_em': 16.666666666666664, 'ovscore': 18.07753815155468}
#  80%|████████  | 4/5 [04:12<01:03, 63.23s/it]

In [ ]:
scores_df=pd.DataFrame(scores_list)

In [ ]:
scores_df.mean()

# rougeLsum    29.178478
# length       93.666667
# str_em       30.555556
# ovscore      18.200875
# dtype: float64
